# 12 — Ensemble Methods & Model Explainability (SHAP)
Train Random Forest and XGBoost, compare against previous models,
then use SHAP to explain individual predictions.

In [ ]:
import pandas as pd
import numpy as np
import pickle
import os
import warnings
warnings.filterwarnings("ignore")

from sklearn.ensemble import RandomForestClassifier
from xgboost import XGBClassifier
from sklearn.pipeline import Pipeline
from sklearn.model_selection import RandomizedSearchCV
from sklearn.metrics import (f1_score, precision_score, recall_score,
                             accuracy_score, classification_report,
                             ConfusionMatrixDisplay)
from scipy.stats import randint, uniform
import shap
import matplotlib.pyplot as plt

from src.preprocess import build_preprocessor
from src.config import (NUMERICAL_FEATURES, CATEGORICAL_FEATURES,
                        RANDOM_STATE, MODEL_PATH, RESULTS_PATH)

plt.rcParams['figure.figsize'] = (12, 8)

## 1. Load Train-Test Data

In [ ]:
X_train = pd.read_csv('data/processed/X_train.csv')
X_test = pd.read_csv('data/processed/X_test.csv')
y_train = pd.read_csv('data/processed/y_train.csv').values.ravel()
y_test = pd.read_csv('data/processed/y_test.csv').values.ravel()
print(f"Train: {X_train.shape}, Test: {X_test.shape}")
print(f"Class distribution (train): {np.bincount(y_train)}")
print(f"Default rate: {y_train.mean()*100:.1f}%")

## 2. Random Forest
Random Forest = many Decision Trees voting together.
Each tree sees a random subset of data and features,
then the final prediction is the majority vote.

**Why better than a single Decision Tree?**
- Individual trees overfit → averaging reduces variance
- Different trees learn different patterns → ensemble catches more

In [ ]:
rf_pipeline = Pipeline(steps=[
    ('preprocessor', build_preprocessor()),
    ('classifier', RandomForestClassifier(random_state=RANDOM_STATE, n_jobs=-1))
])

rf_params = {
    'classifier__n_estimators': [100, 200, 300],
    'classifier__max_depth': [5, 7, 10, None],
    'classifier__min_samples_split': randint(2, 10),
    'classifier__class_weight': ['balanced', None]
}

rf_search = RandomizedSearchCV(
    rf_pipeline, rf_params, n_iter=15, cv=3,
    scoring='f1', random_state=RANDOM_STATE, n_jobs=-1, verbose=1
)
rf_search.fit(X_train, y_train)

print(f"\nBest params: {rf_search.best_params_}")
print(f"Best CV F1: {rf_search.best_score_:.4f}")

y_pred_rf = rf_search.predict(X_test)
print(f"\nTest F1 (class 1): {f1_score(y_test, y_pred_rf, pos_label=1):.4f}")
print(classification_report(y_test, y_pred_rf, target_names=['No Default', 'Default']))

## 3. XGBoost
XGBoost = boosted trees. Instead of parallel voting (Random Forest),
each new tree corrects the mistakes of the previous tree.

**Why XGBoost dominates Kaggle competitions:**
- Handles missing values natively
- Built-in regularization (L1/L2) prevents overfitting
- `scale_pos_weight` handles class imbalance without SMOTE

In [ ]:
# Calculate class imbalance ratio for XGBoost
scale_pos = np.sum(y_train == 0) / np.sum(y_train == 1)
print(f"scale_pos_weight = {scale_pos:.2f} (ratio of negatives to positives)")

xgb_pipeline = Pipeline(steps=[
    ('preprocessor', build_preprocessor()),
    ('classifier', XGBClassifier(
        random_state=RANDOM_STATE,
        eval_metric='logloss',
        n_jobs=-1,
        use_label_encoder=False
    ))
])

xgb_params = {
    'classifier__n_estimators': [100, 200, 300],
    'classifier__max_depth': [3, 5, 7],
    'classifier__learning_rate': [0.01, 0.05, 0.1, 0.2],
    'classifier__scale_pos_weight': [1, scale_pos],
    'classifier__subsample': [0.7, 0.8, 1.0]
}

xgb_search = RandomizedSearchCV(
    xgb_pipeline, xgb_params, n_iter=20, cv=3,
    scoring='f1', random_state=RANDOM_STATE, n_jobs=-1, verbose=1
)
xgb_search.fit(X_train, y_train)

print(f"\nBest params: {xgb_search.best_params_}")
print(f"Best CV F1: {xgb_search.best_score_:.4f}")

y_pred_xgb = xgb_search.predict(X_test)
print(f"\nTest F1 (class 1): {f1_score(y_test, y_pred_xgb, pos_label=1):.4f}")
print(classification_report(y_test, y_pred_xgb, target_names=['No Default', 'Default']))

## 4. Ensemble vs Previous Models — Comparison

In [ ]:
models = {
    'Random Forest': (rf_search, y_pred_rf),
    'XGBoost': (xgb_search, y_pred_xgb),
}

# Load previous results if they exist
if os.path.exists(RESULTS_PATH):
    prev_results = pd.read_csv(RESULTS_PATH)
    print("Previous model results loaded.")
else:
    prev_results = pd.DataFrame()

new_rows = []
for name, (search, y_pred) in models.items():
    new_rows.append({
        'Model': name,
        'Accuracy': accuracy_score(y_test, y_pred),
        'Precision (class 1)': precision_score(y_test, y_pred, pos_label=1),
        'Recall (class 1)': recall_score(y_test, y_pred, pos_label=1),
        'Test F1 (class 1)': f1_score(y_test, y_pred, pos_label=1),
    })

new_df = pd.DataFrame(new_rows)
all_results = pd.concat([prev_results, new_df], ignore_index=True).round(4)
all_results = all_results.sort_values('Test F1 (class 1)', ascending=False)
all_results.to_csv(RESULTS_PATH, index=False)
print(all_results.to_string(index=False))

## 5. Confusion Matrices

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(14, 5))
for ax, (name, (_, y_pred)) in zip(axes, models.items()):
    ConfusionMatrixDisplay.from_predictions(y_test, y_pred,
        display_labels=['No Default', 'Default'], ax=ax, cmap='Blues')
    ax.set_title(f'{name}', fontsize=14, fontweight='bold')
plt.tight_layout()
plt.savefig('docs/eda_plots/ensemble_confusion_matrices.png', dpi=150, bbox_inches='tight')
plt.show()

## 6. Save Best Ensemble Model

In [ ]:
rf_f1 = f1_score(y_test, y_pred_rf, pos_label=1)
xgb_f1 = f1_score(y_test, y_pred_xgb, pos_label=1)

if xgb_f1 >= rf_f1:
    best_pipeline = xgb_search.best_estimator_
    best_name = "XGBoost"
    best_f1 = xgb_f1
else:
    best_pipeline = rf_search.best_estimator_
    best_name = "Random Forest"
    best_f1 = rf_f1

print(f"Best model: {best_name} (F1={best_f1:.4f})")

os.makedirs('models', exist_ok=True)
with open(MODEL_PATH, 'wb') as f:
    pickle.dump(best_pipeline, f)
print(f"Saved to {MODEL_PATH}")

## 7. SHAP — Model Explainability

**SHAP (SHapley Additive exPlanations)** explains individual predictions.
For each prediction, SHAP tells you exactly how much each feature
pushed the prediction toward "Default" or "No Default."

**Why this matters for lending:**
- Regulators require explainable decisions (ECOA, GDPR)
- Borrowers have a right to know WHY they were rejected
- Banks need to trust the model before deploying it

In [ ]:
# Get the preprocessor and transform the data
preprocessor = best_pipeline.named_steps['preprocessor']
classifier = best_pipeline.named_steps['classifier']

X_train_transformed = preprocessor.transform(X_train)
X_test_transformed = preprocessor.transform(X_test)

# Get feature names after one-hot encoding
num_features = NUMERICAL_FEATURES
try:
    cat_features = list(preprocessor.named_transformers_['cat']
                        .named_steps['encoder']
                        .get_feature_names_out(CATEGORICAL_FEATURES))
except:
    cat_features = [f"cat_{i}" for i in range(X_train_transformed.shape[1] - len(num_features))]
feature_names = num_features + cat_features
print(f"Total features after encoding: {len(feature_names)}")

### 7a. Global Feature Importance (SHAP Summary Plot)

In [ ]:
# Use TreeExplainer for tree-based models (fast)
explainer = shap.TreeExplainer(classifier)

# Explain a sample of test predictions (100 rows for speed)
sample_idx = np.random.choice(len(X_test_transformed), size=100, replace=False)
X_sample = X_test_transformed[sample_idx]
shap_values = explainer.shap_values(X_sample)

# For binary classification, shap_values might be a list of 2 arrays
# We want class 1 (Default)
if isinstance(shap_values, list):
    shap_vals = shap_values[1]  # class 1
else:
    shap_vals = shap_values

fig, ax = plt.subplots(figsize=(12, 8))
shap.summary_plot(shap_vals, X_sample, feature_names=feature_names, show=False)
plt.title('SHAP Summary — Which Features Drive Default Predictions?', fontsize=14, fontweight='bold')
plt.tight_layout()
plt.savefig('docs/eda_plots/shap_summary.png', dpi=150, bbox_inches='tight')
plt.show()

### 7b. Single Prediction Explanation (Waterfall Plot)
This shows exactly WHY the model predicted a specific borrower as high-risk or low-risk.

In [ ]:
# Pick a defaulter from the test set
default_indices = np.where(y_test == 1)[0]
example_idx = default_indices[0]

example_transformed = preprocessor.transform(X_test.iloc[[example_idx]])
example_shap = explainer.shap_values(example_transformed)

if isinstance(example_shap, list):
    example_shap_vals = example_shap[1][0]
else:
    example_shap_vals = example_shap[0]

print(f"Actual label: Default ({y_test[example_idx]})")
print(f"Model prediction: {best_pipeline.predict(X_test.iloc[[example_idx]])[0]}")
print(f"Default probability: {best_pipeline.predict_proba(X_test.iloc[[example_idx]])[0][1]:.4f}")
print(f"\nBorrower profile:")
for col in NUMERICAL_FEATURES + CATEGORICAL_FEATURES:
    print(f"  {col}: {X_test.iloc[example_idx][col]}")

# Waterfall plot
explanation = shap.Explanation(
    values=example_shap_vals,
    base_values=explainer.expected_value[1] if isinstance(explainer.expected_value, (list, np.ndarray)) else explainer.expected_value,
    data=example_transformed[0],
    feature_names=feature_names
)
fig, ax = plt.subplots(figsize=(12, 8))
shap.plots.waterfall(explanation, show=False)
plt.title('Why did the model flag this borrower?', fontsize=14, fontweight='bold')
plt.tight_layout()
plt.savefig('docs/eda_plots/shap_waterfall.png', dpi=150, bbox_inches='tight')
plt.show()

## 8. Key Takeaways

### Ensemble Methods
- **Random Forest:** Parallel trees → reduces variance, robust to noise
- **XGBoost:** Sequential correction → reduces bias, industry standard for tabular data
- Both significantly outperform individual Decision Trees and Logistic Regression

### SHAP Explainability
- Every prediction can be explained with specific feature contributions
- Global summary shows which features matter MOST across all predictions
- Waterfall plot shows WHY a specific borrower was flagged
- This is not optional in regulated industries — it is required by law (ECOA, GDPR)